**Finesse — Gamified Budgeting & Financial Health Scorer**

Anggota:
1. CFCC319D6Y0190 - Patrick Nicxon Hutabarat - Full-Stack Web Developer
2. CDCC319D6X0998 - Dame Theresia Rejeki Sidauruk - Data Science
3. CDCC319D6X1254 - Cikita Natasya Br Sembiring - Data Scientist
4. CACC319D6Y0343 - Rayza Indafri Yahya - AI Engineer
5. CACC319D6Y1720 - Samuel Gautama Manik - AI Engineer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Finesse: Gamified Budgeting & Financial Health Scorer**


### Tahap 1: Data Wrangling & Pembersihan Awal

 **Langkah 1: Inisialisasi dan Memuat Data**

Pada tahap ini, kita akan menghubungkan Google Colab dengan Google Drive untuk mengakses dataset asli secara persisten, lalu memuatnya ke dalam Pandas DataFrame.

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

# 1. Menghubungkan ke Google Drive
drive.mount('/content/drive')

# 2. Membaca dataset asli (Pastikan file sudah di-upload ke Drive)
file_path = '/content/drive/MyDrive/Finesse_Project/UrbanMart_Transactions.csv'
df = pd.read_csv(file_path)

print(f"Dataset berhasil dimuat. Total data asli: {len(df)} baris.")
df.head(3)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset berhasil dimuat. Total data asli: 50000 baris.


,TransactionID,CustomerID,TransactionDate,TransactionValue,ProductCategory,PaymentMethod,CustomerGender,CustomerAgeGroup,Region
0,TRANS000000,CUST01574,2023-04-09,467907,Sport,Bank Transfer,Female,>50,Surabaya
1,TRANS000001,CUST01836,2024-08-05,2713789,Beauty,Credit Card,Female,26-35,Surabaya
2,TRANS000002,CUST01291,2024-04-07,4539248,Fashion,Bank Transfer,Male,18-25,Bandung


 **Langkah 2 : Memfilter Target Pengguna dan Standarisasi**


Mengingat Finesse adalah aplikasi yang ditujukan untuk generasi muda (khususnya mahasiswa), kita perlu menyaring data dan hanya mempertahankan transaksi dari pelanggan dengan kelompok usia 18-25 tahun. Selanjutnya, dilakukan standarisasi nama kolom agar sesuai dengan *Data Dictionary* yang telah disepakati oleh tim *Engineering*.

In [ ]:
# Memfilter populasi berdasarkan usia target (18-25 tahun)
df_student = df[df['CustomerAgeGroup'] == '18-25'].copy()
print(f"Total data setelah pemfilteran demografi: {len(df_student)} baris.")

# Melakukan standarisasi (rename) kolom untuk konsistensi sistem
df_student = df_student.rename(columns={
    'TransactionID': 'transaction_id',
    'CustomerID': 'user_id',
    'TransactionDate': 'date',
    'TransactionValue': 'amount',
    'ProductCategory': 'category',
    'PaymentMethod': 'payment_method'
})
df_student.head(2)

Total data setelah pemfilteran demografi: 15239 baris.


,transaction_id,user_id,date,amount,category,payment_method,CustomerGender,CustomerAgeGroup,Region
2,TRANS000002,CUST01291,2024-04-07,4539248,Fashion,Bank Transfer,Male,18-25,Bandung
5,TRANS000005,CUST00331,2024-11-24,1556151,Electronics,Credit Card,Male,18-25,Bandung


### Tahap 2: Transformasi Data Sintetis (Synthetic Data Generation)

**Langkah 3: Personalisasi Kategori Pengeluaran**


Data publik asli berisi kategori belanja supermarket umum. Untuk menciptakan lingkungan simulasi yang akurat bagi model gamifikasi Finesse, kategori tersebut ditransformasikan menjadi 6 kategori pengeluaran khas mahasiswa. Probabilitas kemunculan (distribusi) untuk setiap kategori juga diatur agar mencerminkan realita pola pengeluaran harian.

In [ ]:
# Mendefinisikan kategori relevan untuk mahasiswa
student_categories = ['Makan & Minum', 'Transportasi', 'Tagihan & Kos', 'Kebutuhan Kuliah', 'Hiburan & Nongkrong', 'Belanja Umum']

# Mengatur random seed demi reproduktibilitas (reproducibility) eksperimen
np.random.seed(42)

# Mengganti nilai kolom secara acak (berbobot)
df_student['category'] = np.random.choice(
    student_categories,
    size=len(df_student),
    p=[0.40, 0.20, 0.05, 0.10, 0.15, 0.10]
)

**Langkah 4: Penyesuaian Skala Ekonomi (Nominal Rupiah)**


Nominal transaksi asli (*TransactionValue*) tidak merepresentasikan daya beli rata-rata mahasiswa. Tahap ini menurunkan skala *amount* tersebut ke rentang yang lebih logis, disesuaikan secara dinamis berdasarkan masing-masing kategori pengeluarannya.

In [ ]:
# Fungsi untuk mereskalasi nilai transaksi per kategori (dalam Rupiah)
def adjust_amount(cat):
    if cat == 'Makan & Minum':
        return np.random.randint(15, 75) * 1000
    elif cat == 'Transportasi':
        return np.random.randint(10, 50) * 1000
    elif cat == 'Tagihan & Kos':
        return np.random.randint(500, 2500) * 1000
    elif cat == 'Kebutuhan Kuliah':
        return np.random.randint(20, 200) * 1000
    elif cat == 'Hiburan & Nongkrong':
        return np.random.randint(50, 300) * 1000
    else:
        return np.random.randint(50, 500) * 1000

# Menerapkan transformasi (apply) pada kolom 'amount'
df_student['amount'] = df_student['category'].apply(adjust_amount)

# Membuang kolom bawaan dataset asli yang tidak relevan (Feature Selection)
final_columns = ['transaction_id', 'user_id', 'date', 'amount', 'category', 'payment_method']
df_final = df_student[final_columns]

### Tahap 3: Feature Engineering (Persiapan Target AI)


**Langkah 5: Simulasi Profil Pengguna (Alokasi Budget)**


Algoritma *Deep Neural Network* (DNN) Finesse membutuhkan konteks untuk menilai perilaku finansial. Pengeluaran Rp 1.000.000 hanya dapat dinilai "boros" atau "hemat" jika kita mengetahui total anggarannya. Proses ini mengalokasikan anggaran bulanan acak kepada setiap pengguna unik.

In [ ]:
# Mengekstrak populasi pengguna unik
unique_users = df_final['user_id'].unique()

# Mendistribusikan alokasi budget bulanan (Rp 1.5 Jt - Rp 4 Jt)
np.random.seed(42)
user_budgets = np.random.randint(1500, 4000, size=len(unique_users)) * 1000

# Membangun relasi (merge) antara profil pengguna dengan log transaksi
df_profiles = pd.DataFrame({
    'user_id': unique_users,
    'monthly_budget': user_budgets
})
df_ai_ready = df_final.merge(df_profiles, on='user_id', how='left')

print("Integrasi budget bulanan berhasil dilakukan:")
df_ai_ready[['user_id', 'amount', 'monthly_budget']].head(3)

Integrasi budget bulanan berhasil dilakukan:


,user_id,amount,monthly_budget
0,CUST01291,19000,2360000
1,CUST00331,102000,2794000
2,CUST00112,192000,2630000


**Langkah 6: Menghitung Akumulasi Beban Finansial (Cumulative Spend)**


Untuk memprediksi kesehatan finansial *real-time* (pada saat transaksi dilakukan), model membutuhkan variabel beban kumulatif. Proses agregasi ini menghitung total pengeluaran kumulatif dari awal bulan hingga tanggal transaksi (YTD/Month-to-Date).

In [ ]:
# Mengonversi format tanggal menjadi objek datetime
df_ai_ready['date'] = pd.to_datetime(df_ai_ready['date'])

# Mengekstrak periode Tahun-Bulan untuk proses agregasi
df_ai_ready['year_month'] = df_ai_ready['date'].dt.to_period('M')

# Mengurutkan dataset secara kronologis berdasarkan pengguna
df_ai_ready = df_ai_ready.sort_values(by=['user_id', 'date'])

# Menghitung agregasi pengeluaran kumulatif (cumsum) per periode
df_ai_ready['cumulative_spend'] = df_ai_ready.groupby(['user_id', 'year_month'])['amount'].cumsum()

print("Agregasi beban kumulatif per transaksi:")
df_ai_ready[['user_id', 'date', 'amount', 'cumulative_spend']].head(3)

Agregasi beban kumulatif per transaksi:


,user_id,date,amount,cumulative_spend
10380,CUST00001,2023-03-14,179000,179000
7201,CUST00001,2023-03-24,29000,208000
825,CUST00001,2023-04-14,480000,480000


**Langkah 7: Penetapan Variabel Target (Financial Health Score)**


Langkah puncak dalam rekayasa fitur ini adalah merumuskan variabel `financial_health_score`. Fitur turunan ini akan difungsikan sebagai *Target Label* dalam pelatihan model regresi *Deep Learning*. Skor diinisialisasi dari angka 100 dan akan terdegradasi secara proporsional seiring dengan menipisnya rasio anggaran bulanan.

In [ ]:
# Mengukur rasio konsumsi anggaran (dalam persentase)
spend_ratio = (df_ai_ready['cumulative_spend'] / df_ai_ready['monthly_budget']) * 100

# Menghitung skor akhir (bounding antara 0 hingga 100)
df_ai_ready['financial_health_score'] = np.clip(100 - spend_ratio, 0, 100)
df_ai_ready['financial_health_score'] = df_ai_ready['financial_health_score'].round(2)

print("Kalkulasi Target Label Model AI:")
df_ai_ready[['user_id', 'monthly_budget', 'cumulative_spend', 'financial_health_score']].head(5)

Kalkulasi Target Label Model AI:


,user_id,monthly_budget,cumulative_spend,financial_health_score
10380,CUST00001,1614000,179000,88.91
7201,CUST00001,1614000,208000,87.11
825,CUST00001,1614000,480000,70.26
3918,CUST00001,1614000,532000,67.04
2288,CUST00001,1614000,45000,97.21


**Langkah 8: Eksportase Dataset Final**


Data sekarang terstruktur secara definitif. Kolom komputasi sementara dieliminasi untuk meringankan dimensi data, dan *dataset* dikonversikan ke dalam format `.csv` final yang siap dieksekusi oleh tim divisi AI (*Artificial Intelligence*) dan EDA.

In [ ]:
# Eliminasi kolom agregasi sementara (menggunakan errors='ignore' agar aman jika di-run berkali-kali)
df_ai_ready = df_ai_ready.drop(columns=['year_month'], errors='ignore')

# Eksekusi ekspor ke dalam penyimpanan Google Drive
nama_file_ai = '/content/drive/MyDrive/Finesse_Project/finesse_dataset.csv'
df_ai_ready.to_csv(nama_file_ai, index=False)

print(f"Data Wrangling Selesai! Dataset final berisi {len(df_ai_ready)} baris berhasil disimpan.")

Data Wrangling Selesai! Dataset final berisi 15239 baris berhasil disimpan.


## 4. Daftar Pertanyaan Bisnis
Berdasarkan dataset yang telah dirancang, berikut adalah daftar pertanyaan komprehensif yang akan dijawab pada tahap *Exploratory Data Analysis* (EDA):

1. **Analisis Komposisi:** Kategori pengeluaran manakah yang mendominasi persentase total alokasi pengeluaran bulanan (Share of Wallet) mahasiswa?
2. **Analisis Temporal (Tren Penurunan):** Pada interval waktu mingguan manakah mahasiswa paling rentan mengalami degradasi tingkat kesehatan finansial (*Financial Health Score*) yang paling drastis?
3. **Analisis Korelasi Davian (Digital Payment):** Apakah penggunaan instrumen pembayaran digital (E-Wallet/Bank Transfer/Credit Card) memiliki korelasi empiris terhadap tingginya nominal pengeluaran (*ticket size*) per transaksi?
4. **Analisis Segmentasi Kluster:** Bagaimana divergensi proporsi antara belanja primer (Kebutuhan Pokok) dengan sekunder (Hiburan & Nongkrong) memetakan profil risiko konsumtif per pengguna?